# Geometry-V5 M0 small Colab diagnostic

This handoff runs a four-case, nonformal method diagnostic only. Its science denominator is 0; it is not a RELIABLE, robustness, crop, or science gate.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clone the fixed source

The checkout and install are deliberately single-path: a fresh clone, detached exact, and one installation.

In [ ]:
import subprocess
import sys
from pathlib import Path

SOURCE_EXACT = 'd193d0fe39702e373e087ca3e9a64e8d7f3cbfef'
REPO = Path('/content/geometry-v5-m0-diagnostic')
if REPO.exists():
    raise RuntimeError('fresh checkout path already exists')

quiet = {'stdout': subprocess.DEVNULL, 'stderr': subprocess.DEVNULL}
subprocess.run(['git', 'clone', '--branch', 'Geometry-V5', '--single-branch', 'https://github.com/RICHAAARC/CEG-WM.git', str(REPO)], check=True, **quiet)
subprocess.run(['git', 'checkout', '--detach', SOURCE_EXACT], cwd=REPO, check=True, **quiet)

def git_output(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout.strip()

if git_output('rev-parse', 'HEAD') != SOURCE_EXACT:
    raise RuntimeError('detached source exact differs')
if git_output('branch', '--show-current') != '':
    raise RuntimeError('checkout is not detached')
if git_output('status', '--porcelain') != '':
    raise RuntimeError('checkout is not clean')
subprocess.run([sys.executable, '-m', 'pip', 'install', '.'], cwd=REPO, check=True, **quiet)

## 2. Check CUDA and run the diagnostic

Drive is used only for one create-only JSON result. If CUDA is unavailable, stop before invoking the runner and do not produce a diagnostic result.

In [ ]:
import json
from datetime import datetime, timezone

import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required before the M0 diagnostic runner')

OUTPUT_ROOT = Path('/content/drive/MyDrive/CEG-WM/Geometry-V5/M0-Diagnostic/runs')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = OUTPUT_ROOT / f'{SOURCE_EXACT}_seed7501_{timestamp}.json'
if OUTPUT.exists():
    raise FileExistsError('diagnostic JSON already exists')

completed = subprocess.run(
    [sys.executable, '-m', 'experiments.geometry_v5_m0_diagnostic', '--repo-root', str(REPO), '--output-json', str(OUTPUT)],
    cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, check=False,
)
if completed.returncode != 0:
    try:
        diagnostic = json.loads(OUTPUT.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        raise RuntimeError('diagnostic incomplete; result JSON unreadable')
    cases = diagnostic.get('cases')
    if isinstance(cases, list):
        for case in cases:
            if not isinstance(case, dict):
                continue
            report = {}
            for field in ('attack_id', 'failure_stage', 'error_class'):
                if field in case:
                    report[field] = case[field]
            raw = case.get('raw')
            if isinstance(raw, dict):
                if 'status' in raw:
                    report['raw.status'] = raw['status']
                if 'diagnostics' in raw:
                    report['diagnostics'] = raw['diagnostics']
            print(json.dumps(report, ensure_ascii=False, sort_keys=True, separators=(',', ':')))
    raise RuntimeError('diagnostic incomplete; per-case status printed')

lines = [line for line in completed.stdout.splitlines() if line.strip()]
if not lines:
    raise RuntimeError('diagnostic emitted no summary')
summary_line = lines[-1]
if len(summary_line.encode('utf-8')) > 4096:
    raise RuntimeError('diagnostic summary exceeds bound')
summary = json.loads(summary_line)
print(json.dumps(summary, ensure_ascii=False, sort_keys=True, separators=(',', ':')))

## Interpretation

The JSON contains only the small diagnostic result. It does not establish positive watermark evidence, RELIABLE geometry, robustness, crop handling, or scientific success.